In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42); np.random.seed(42)

# 1. Load and clean
df = pd.read_excel("P1_Manufacturing_Quality_Inspection_Dataset.xlsx")
print(df.shape); print(df["Quality_Status"].value_counts(normalize=True))
df = df.drop(columns=["Batch_ID", "Defect_Type"])        # ID + leakage column
y = (df.pop("Quality_Status") == "Fail").astype(int)     # 1 = Fail (defect), 0 = Pass
X = df

cat_cols = ["Machine_ID", "Shift"]
num_cols = [c for c in X.columns if c not in cat_cols]

# 2. Split first, then fit the preprocessor on train only
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

prep = ColumnTransformer([("num", StandardScaler(), num_cols),
                          ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])
Xtr = prep.fit_transform(X_train).astype("float32")
Xva = prep.transform(X_val).astype("float32")
Xte = prep.transform(X_test).astype("float32")

# 3. ANN architecture
model = keras.Sequential([
    layers.Input(shape=(Xtr.shape[1],)),
    layers.Dense(64, activation="relu"), layers.Dropout(0.3),
    layers.Dense(32, activation="relu"), layers.Dropout(0.2),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
              metrics=["accuracy", keras.metrics.AUC(name="auc")])
model.summary()

cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
             keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4)]
hist = model.fit(Xtr, y_train, validation_data=(Xva, y_val), epochs=100, batch_size=64,
                 class_weight={0: cw[0], 1: cw[1]}, callbacks=callbacks, verbose=2)

# 4. Evaluation
proba = model.predict(Xte).ravel()
pred = (proba >= 0.5).astype(int)
print("Accuracy :", round(accuracy_score(y_test, pred), 4))
print("Precision:", round(precision_score(y_test, pred), 4))
print("Recall   :", round(recall_score(y_test, pred), 4))
print("F1       :", round(f1_score(y_test, pred), 4))
print("ROC-AUC  :", round(roc_auc_score(y_test, proba), 4))
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=["Pass", "Fail"]))

# 5. Learning curves
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(hist.history["loss"], label="train"); ax[0].plot(hist.history["val_loss"], label="val"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(hist.history["accuracy"], label="train"); ax[1].plot(hist.history["val_accuracy"], label="val"); ax[1].set_title("Accuracy"); ax[1].legend()
plt.tight_layout(); plt.show()

# 6. Key defect drivers: permutation importance (drop in AUC when a feature is shuffled)
base = roc_auc_score(y_test, proba)
imp = {}
for col in X_test.columns:
    drops = []
    for _ in range(5):
        Xs = X_test.copy(); Xs[col] = np.random.permutation(Xs[col].values)
        drops.append(base - roc_auc_score(y_test, model.predict(prep.transform(Xs).astype("float32"), verbose=0).ravel()))
    imp[col] = np.mean(drops)
imp = pd.Series(imp).sort_values(ascending=False)
print(imp)
imp.head(10)[::-1].plot(kind="barh", title="Permutation importance (AUC drop)"); plt.tight_layout(); plt.show()

# 7. Business view: failure rate by key drivers
raw = pd.read_excel("P1_Manufacturing_Quality_Inspection_Dataset.xlsx")
raw["Fail"] = (raw["Quality_Status"] == "Fail").astype(int)
for col in ["Tool_Wear", "Spindle_Speed", "Vibration_Level"]:
    print(raw.groupby(pd.qcut(raw[col], 5))["Fail"].mean())
print(raw.groupby("Defect_Type").size())